In [7]:
%pip install pdfplumber

import pdfplumber
import pandas as pd


# 1. Load PDF
#------------


pdf_path = r"\CTO2025RESULTS.PDF"

rows = []

with pdfplumber.open(pdf_path) as pdf:
    for page in pdf.pages:
        table = page.extract_table()

        if table:
            rows.extend(table)

df = pd.DataFrame(rows)


# 2. Inspect raw extraction
# -----------------------------

print(df.shape)
display(df.head(15))
display(df.tail(15))
df.info()


# 3. Assign column names
# ----------------------

columns = [
    "OVR",
    "NAME",
    "HOMETOWN",
    "DIVISION",
    "TIME",
    "PACE",
    "GENDER_RANK",
    "DIV_RANK",
    "BIB"
]

df = df.iloc[:, :9]
df.columns = columns


# 4. Remove repeated headers
# -----------------------------

df = df[df["OVR"].notna()]
df = df[df["OVR"] != "OVR"]

# -----------------------------
# 5. Convert numeric columns
# -----------------------------

numeric_cols = [
    "OVR",
    "GENDER_RANK",
    "DIV_RANK",
    "BIB"
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# -----------------------------
# 6. Create gender variable
# -----------------------------

df["GENDER"] = df["DIVISION"].str.extract(
    r"^(Male|Female)",
    expand=False
)

# -----------------------------
# 7. Create age-group variable
# -----------------------------

df["AGE_GROUP"] = (
    df["DIVISION"]
    .str.replace(r"^(Male|Female)", "", regex=True)
    .str.strip()
)

# Standardize youngest age group
df["AGE_GROUP"] = df["AGE_GROUP"].replace({
    "19&Under": "19 & Under"
})

# -----------------------------
# 8. Convert race time to minutes
# -----------------------------

def time_to_minutes(x):
    try:
        h, m, s = str(x).split(":")
        return float(h) * 60 + float(m) + float(s) / 60
    except (ValueError, AttributeError):
        return None

df["TIME_MINUTES"] = df["TIME"].apply(time_to_minutes)

# -----------------------------
# 9. Clean hometown text
# -----------------------------

df["HOMETOWN"] = (
    df["HOMETOWN"]
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
    .str.title()
)

# Show hometown values before manual corrections
print("\nUnique hometowns:")
print(sorted(df["HOMETOWN"].dropna().unique()))

print("\nHometown frequencies:")
print(
    df["HOMETOWN"]
    .value_counts()
    .sort_index()
    .to_string()
)

# -----------------------------
# 10. Fix known hometown typos
# -----------------------------

city_corrections = {
    "Losangelescommerce,Ca": "Los Angeles"
}

df["HOMETOWN"] = df["HOMETOWN"].replace(city_corrections)

# -----------------------------
# 11. Validation checks
# -----------------------------

print("\nGender values:")
print(df["GENDER"].value_counts(dropna=False))

print("\nAge group values:")
print(df["AGE_GROUP"].value_counts(dropna=False).sort_index())

print("\nDivision / Gender / Age Group check:")
display(
    df[
        ["DIVISION", "GENDER", "AGE_GROUP"]
    ].head(20)
)

print("\nMissing values:")
print(df.isna().sum())

print("\nDuplicate BIB numbers:")
print(df["BIB"].duplicated().sum())

print("\nTime summary:")
print(df["TIME_MINUTES"].describe())

# -----------------------------
# 12. Export cleaned dataset
# -----------------------------

df.to_csv(
    r"\CTO2025_clean.csv",
    index=False
)

print("\nCTO2025_clean.csv successfully saved.")


Note: you may need to restart the kernel to use updated packages.
(262, 9)


,0,1,2,3,4,5,6,7,8
0,OVR,NAME,HOMETOWN,DIVISION,FullCourseNetTime,PACE,GENDER_RANK,DIV_RANK,BIB
1,1,CASEYMORRISON,"LosAngeles,CA",Male20-29,00:22:47.04,07:21,1,1,8
2,2,MELCHICEDECPALACIOS,"Hawthorne,CA",Male40-49,00:23:34.31,07:36,2,1,17
3,3,JUANGARCIA,"Rimforest,CA",Male50-59,00:25:35.38,08:15,3,1,75
4,4,ARIELRODRIGUEZ,"Palmdale,CA",Male40-49,00:26:27.66,08:32,4,2,113
5,5,AIDENPALACIOS,"SouthGate,CA",Male19&Under,00:27:05.08,08:44,5,1,209
6,6,VICTORHUGOAGUIN,"LosAngeles,CA",Male40-49,00:27:23.51,08:49,7,3,193
7,7,JOSIAHPOLHEMUS,"Inglewood,CA",Male20-29,00:27:24.31,08:50,6,2,258
8,8,DAVISLOUIE,"Alhambra,CA",Male30-39,00:28:16.29,09:06,8,1,125
9,9,COSETTESMITH,"CulverCity,CA",Female40-49,00:28:28.17,09:10,1,1,164


,0,1,2,3,4,5,6,7,8
247,241,ARACELYGALICIA,"LosAngeles,CA",Female40-49,01:08:29.66,22:03,129,35,143
248,242,JOHNETTAHALL,"CanyonCountry,CA",Female20-29,01:09:14.13,22:18,131,21,134
249,243,SALLYPEREIRA,"-,US",Female30-39,01:09:52.90,22:30,132,37,63
250,244,GINACALIBOSO,"LosAngeles,CA",Female50-59,01:10:04.31,22:34,133,24,126
251,245,ERIKAVIDAL,"NorthHollywood,CA",Female40-49,01:12:05.04,23:13,135,36,214
252,246,ELIZABETHOROZCO,"-,US",Female30-39,01:12:05.08,23:13,134,38,37
253,247,JAHAYRAGALINDO,"-,US",Female30-39,01:12:20.66,23:18,136,39,46
254,248,BRITTANINORA,"Carson,CA",Female30-39,01:12:58.96,23:30,137,40,202
255,249,ALISHAALVAREZ,"SanFernando,CA",Female40-49,01:14:15.96,23:55,138,37,78
256,250,LECHELLEJONES,"LaPalma,CA",Female30-39,01:14:30.17,23:59,139,41,203


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 262 entries, 0 to 261
Data columns (total 9 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   0       262 non-null    object
 1   1       262 non-null    object
 2   2       262 non-null    object
 3   3       262 non-null    object
 4   4       262 non-null    object
 5   5       262 non-null    object
 6   6       262 non-null    object
 7   7       262 non-null    object
 8   8       262 non-null    object
dtypes: object(9)
memory usage: 18.6+ KB

Unique hometowns:
['-,Us', 'Alhambra,Ca', 'Baldwinpark,Ca', 'Banning,Ca', 'Bellgardens,Ca', 'Burbank,Ca', 'Canogapark,Ca', 'Canyoncountry,Ca', 'Carson,Ca', 'Compton,Ca', 'Culvercity,Ca', 'Downey,Ca', 'Fontana,Ca', 'Gainesville,Fl', 'Gardena,Ca', 'Glendale,Ca', 'Granadahills,Ca', 'Hawthorne,Ca', 'Huntingtonbeach,Ca', 'Inglewood,Ca', 'Lacanadaflintridge,Ca', 'Lamirada,Ca', 'Lapalma,Ca', 'Longbeach,Ca', 'Losalamitos,Ca', 'Losangeles,Ca', 'Losangelescommerce

,DIVISION,GENDER,AGE_GROUP
1,Male20-29,Male,20-29
2,Male40-49,Male,40-49
3,Male50-59,Male,50-59
4,Male40-49,Male,40-49
5,Male19&Under,Male,19 & Under
6,Male40-49,Male,40-49
7,Male20-29,Male,20-29
8,Male30-39,Male,30-39
9,Female40-49,Female,40-49
10,Male40-49,Male,40-49



Missing values:
OVR             0
NAME            0
HOMETOWN        0
DIVISION        0
TIME            0
PACE            0
GENDER_RANK     0
DIV_RANK        0
BIB             0
GENDER          0
AGE_GROUP       0
TIME_MINUTES    0
dtype: int64

Duplicate BIB numbers:
0

Time summary:
count    255.000000
mean      48.111304
std       11.578594
min       22.784000
25%       40.460083
50%       47.109500
75%       55.939250
max       79.709333
Name: TIME_MINUTES, dtype: float64


PermissionError: [Errno 13] Permission denied: 'C:\\Users\\garci\\OneDrive\\Desktop\\CTO2025_clean.csv'